In [1]:
import re

UNDERLINED = re.compile(r"^<u>(.{3,90}?)</u>$")

In [2]:
def split_heading_context(text: str) -> tuple[str, list[str]]:
    """Separate the carried-forward heading lines from the chunk body."""
    lines = text.split("\n")
    heading_lines, i = [], 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        heading_lines.append(lines[i].strip())
        i += 1
    return " / ".join(heading_lines), lines[i:]

In [3]:
lines = "## Chapter 1 \n Machine learning\n is a field\n ### Introduction \n It uses data"

In [4]:
split_heading_context(lines)

('## Chapter 1',
 [' Machine learning', ' is a field', ' ### Introduction ', ' It uses data'])

In [6]:
import json
from pathlib import Path

In [9]:
def audit(cache_file: Path) -> None:
    chunks = json.loads(cache_file.read_text(encoding="utf-8"))
    print(len(chunks))
    

In [7]:
import sys

In [17]:
print(Path.cwd().parent)

g:\Telegram Desktop\Investor_intelligence_bot


In [35]:
root_path = Path.cwd().parent
cache_dir = root_path / "data"  / "cache"
files = sorted(cache_dir.glob("*.json"))

if not files:
    print(f"No cache files found in {cache_dir}")
for f in files:
    if f.name.endswith(".partial.json"):
        continue
    audit(f)

No undetected headings found.
chunk 154: <u><u>OPERATING EXPENSES</u></u>


wrongly labeled as: [(154, '<u>OPERATING EXPENSES</u>', '## _More Personal Computing_'), (360, '<u>NOTE 2 — EARNINGS PER SHARE</u>', '## **_Income Taxes – Improvements to Income Tax Disclosures_**'), (403, '<u>NOTE 6 — INVENTORIES</u>', '## **Fair Values of Derivative Instruments**'), (406, '<u>NOTE 7 — PROPERTY AND EQUIPMENT</u>', '## **Fair Values of Derivative Instruments**'), (450, '<u>NOTE 11 — DEBT</u>', '## <u>NOTE 10 — INTANGIBLE ASSETS</u>'), (488, '<u>NOTE 13 — UNEARNED REVENUE</u>', '## **Uncertain Tax Positions**'), (505, '<u>NOTE 15 — CONTINGENCIES</u>', '### **<u>(In millions, except lease term and discount rate)</u>**'), (517, '<u>NOTE 16 — STOCKHOLDERS’ EQUITY</u>', '## **Other Contingencies**'), (531, '<u>NOTE 18 — EMPLOYEE STOCK AND SAVINGS PLANS</u>', '## <u>NOTE 17 — ACCUMULATED OTHER COMPREHENSIVE INCOME (LOSS)</u>')]


chunk 360: <u><u>NOTE 2 — EARNINGS PER SHARE</u></u>


wrongly labeled 

[(154, '<u>OPERATING EXPENSES</u>', '## _More Personal Computing_'), (360, '<u>NOTE 2 — EARNINGS PER SHARE</u>', '## **_Income Taxes – Improvements to Income Tax Disclosures_**'), (403, '<u>NOTE 6 — INVENTORIES</u>', '## **Fair Values of Derivative Instruments**'), (406, '<u>NOTE 7 — PROPERTY AND EQUIPMENT</u>', '## **Fair Values of Derivative Instruments**'), (450, '<u>NOTE 11 — DEBT</u>', '## <u>NOTE 10 — INTANGIBLE ASSETS</u>'), (488, '<u>NOTE 13 — UNEARNED REVENUE</u>', '## **Uncertain Tax Positions**'), (505, '<u>NOTE 15 — CONTINGENCIES</u>', '### **<u>(In millions, except lease term and discount rate)</u>**'), (517, '<u>NOTE 16 — STOCKHOLDERS’ EQUITY</u>', '## **Other Contingencies**'), (531, '<u>NOTE 18 — EMPLOYEE STOCK AND SAVINGS PLANS</u>', '## <u>NOTE 17 — ACCUMULATED OTHER COMPREHENSIVE INCOME (LOSS)</u>')]

In [21]:
def is_undetected_heading(line: str) -> bool:
    """A heading-shaped line that pymupdf4llm left unmarked."""
    s = line.strip()
    m = UNDERLINED.fullmatch(s)
    if not m:
        return False
    inner = m.group(1)
    return bool(
        re.search(r"[A-Za-z]{3,}", inner)
        and not inner.rstrip().endswith((".", ",", ";", ":"))
    )

In [49]:
def audit(cache_file: Path) -> None:
    chunks = json.loads(cache_file.read_text(encoding="utf-8"))
    contexts = [split_heading_context(c["text"])[0] for c in chunks]

    culprits = []

    for i, chunk in enumerate(chunks):
            _, body = split_heading_context(chunk["text"])
            for line in body:
                if is_undetected_heading(line):
                    culprits.append((i, line.strip(), contexts[i]))

    if not culprits:
        print("No undetected headings found.")

    total_affected = 0
    by_context = defaultdict(int)


    for idx, heading, wrong_context in culprits:
            # Everything after this point carrying the stale heading is mislabeled,
            # until the carried context changes.
            affected = 0
            for j in range(idx + 1, len(chunks)):
                if contexts[j] != wrong_context:
                    break
                affected += 1
    
            tables = sum(
                1
                for j in range(idx, idx + affected + 1)
                if chunks[j].get("content_type") == "table"
            )
            total_affected += affected + 1
            by_context[wrong_context] += affected + 1
    
            print(f"\n  chunk {idx}: {heading}")
            print(f"    wrongly labeled as: {wrong_context or '(no heading)'}")
            print(f"    chunks affected: {affected + 1}  (tables: {tables})")
    
    print(f"\n  {'-' * 66}")
    print(f"  undetected headings : {len(culprits)}")
    print(f"  chunks mislabeled   : {total_affected} / {len(chunks)} "
              f"({100 * total_affected / len(chunks):.1f}%)")
    print(f"  distinct wrong labels: {len(by_context)}")
        

In [54]:
root_path = Path.cwd().parent
cache_dir = root_path / "data"  / "cache"

files = sorted(cache_dir.glob("*.json"))

print(files)

if not files:
    print(f"No cache files found in {cache_dir}")



for f in files:
    if f == Path("g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Microsoft_2024_364c4de7f674688d_v6.json"):

        print(f)

        

        audit(f)

[WindowsPath('g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Apple_2024_b9dc49ea295eea1d_v5.json'), WindowsPath('g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Microsoft_2024_364c4de7f674688d_v5.json'), WindowsPath('g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Microsoft_2024_364c4de7f674688d_v6.json'), WindowsPath('g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Tesla_2024_6966e0db9f5191c6_v4.json'), WindowsPath('g:/Telegram Desktop/Investor_intelligence_bot/data/cache/Tesla_2024_9b08571158ef66bc_v5.json')]
g:\Telegram Desktop\Investor_intelligence_bot\data\cache\Microsoft_2024_364c4de7f674688d_v6.json
No undetected headings found.

  ------------------------------------------------------------------
  undetected headings : 0
  chunks mislabeled   : 0 / 573 (0.0%)
  distinct wrong labels: 0


In [23]:
from collections import defaultdict
print(defaultdict(int))

defaultdict(<class 'int'>, {})


In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("data/raw_pdfs/2024_Microsoft.pdf")
Path("data/markdown_docling/2024_Microsoft.md").write_text(
    result.document.export_to_markdown(), encoding="utf-8"
)

NOTES = [
    "OPERATING EXPENSES",
    "NOTE 2 — EARNINGS PER SHARE",
    "NOTE 6 — INVENTORIES",
    "NOTE 7 — PROPERTY AND EQUIPMENT",
    "NOTE 11 — DEBT",
    "NOTE 13 — UNEARNED REVENUE",
    "NOTE 15 — CONTINGENCIES",
    "NOTE 16 — STOCKHOLDERS",
    "NOTE 18 — EMPLOYEE STOCK AND SAVINGS PLANS",
]